In [1]:
# ==========================================
# CELL 1: Setup Paths & Imports
# ==========================================
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from skimage.feature import graycomatrix, graycoprops

BASE_DIR = r"D:\B. Disease Grading"
TRAIN_IMG_DIR = os.path.join(BASE_DIR, "1. Original Images", "a. Training Set")
TEST_IMG_DIR = os.path.join(BASE_DIR, "1. Original Images", "b. Testing Set")

train_images = sorted(list(Path(TRAIN_IMG_DIR).glob("*.jpg")))
test_images = sorted(list(Path(TEST_IMG_DIR).glob("*.jpg")))

print(f"Loaded {len(train_images)} Train images and {len(test_images)} Test images.")

Loaded 413 Train images and 103 Test images.


In [2]:
# ==========================================
# CELL 2: Pure Classical Lesion Segmentation
# ==========================================
def segment_optic_disc(green_ch):
    """Segment Optic Disc using morphological dilation and intensity thresholding."""
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    top_hat = cv2.morphologyEx(green_ch, cv2.MORPH_TOPHAT, kernel)
    
    # Optic disc is among the brightest regions
    _, od_mask = cv2.threshold(top_hat, np.percentile(top_hat, 98), 255, cv2.THRESH_BINARY)
    od_mask = cv2.dilate(od_mask, kernel, iterations=2)
    return od_mask

def extract_lesion_features(img_path):
    """Extracts features strictly from segmented DR lesions (MA, HE, EX)."""
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    
    green_ch = img[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(green_ch)
    
    # 1. Optic Disc Masking (Subtract bright OD to prevent false exudate counts)
    od_mask = segment_optic_disc(enhanced)
    retina_no_od = cv2.bitwise_and(enhanced, enhanced, mask=cv2.bitwise_not(od_mask))
    
    # 2. Hard Exudates (Bright Lesions)
    kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    top_ex = cv2.morphologyEx(retina_no_od, cv2.MORPH_TOPHAT, kernel_small)
    _, ex_mask = cv2.threshold(top_ex, 30, 255, cv2.THRESH_BINARY)
    ex_area = np.sum(ex_mask == 255)
    num_exudates = cv2.connectedComponents(ex_mask)[0] - 1
    
    # 3. Microaneurysms & Hemorrhages (Dark Lesions)
    black_hat = cv2.morphologyEx(retina_no_od, cv2.MORPH_BLACKHAT, kernel_small)
    _, dark_mask = cv2.threshold(black_hat, 15, 255, cv2.THRESH_BINARY)
    
    # Separate Microaneurysms (small circular) from Hemorrhages (larger area)
    contours, _ = cv2.findContours(dark_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    ma_count = 0
    ma_area = 0
    hem_count = 0
    hem_area = 0
    
    for c in contours:
        area = cv2.contourArea(c)
        if area < 3:
            continue
        elif 3 <= area <= 50: # Small dot lesions (Microaneurysms)
            ma_count += 1
            ma_area += area
        else: # Larger blot lesions (Hemorrhages)
            hem_count += 1
            hem_area += area
            
    # 4. GLCM Texture features strictly within retinal region
    glcm = graycomatrix(enhanced, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
    
    return [
        ma_count, ma_area, 
        hem_count, hem_area, 
        num_exudates, ex_area, 
        contrast, homogeneity
    ]

def process_dataset(image_paths):
    cols = ['Image_Name', 'MA_Count', 'MA_Area', 'Hem_Count', 'Hem_Area', 'Ex_Count', 'Ex_Area', 'GLCM_Contrast', 'GLCM_Homogeneity']
    rows = []
    
    for p in image_paths:
        feats = extract_lesion_features(p)
        if feats is not None:
            rows.append([os.path.basename(p)] + feats)
            
    return pd.DataFrame(rows, columns=cols)

print("Segmenting lesions & extracting features for Training Set...")
df_train_lesions = process_dataset(train_images)

print("Segmenting lesions & extracting features for Testing Set...")
df_test_lesions = process_dataset(test_images)

# Save extracted classical features
train_out = os.path.join(BASE_DIR, "train_lesion_features.csv")
test_out = os.path.join(BASE_DIR, "test_lesion_features.csv")

df_train_lesions.to_csv(train_out, index=False)
df_test_lesions.to_csv(test_out, index=False)

print(f"Features saved: {train_out} and {test_out}")

Segmenting lesions & extracting features for Training Set...
Segmenting lesions & extracting features for Testing Set...
Features saved: D:\B. Disease Grading\train_lesion_features.csv and D:\B. Disease Grading\test_lesion_features.csv
